In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv("flipkart_com-ecommerce_sample.csv")
df.head()

,uniq_id,crawl_timestamp,product_url,product_name,product_category_tree,pid,retail_price,discounted_price,image,is_FK_Advantage_product,description,product_rating,overall_rating,brand,product_specifications
0,c2d766ca982eca8304150849735ffef9,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2FF9KEDEFGF,999.0,379.0,"[""http://img5a.flixcart.com/image/short/u/4/a/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
1,7f7036a6d550aaa89d34c77bd39a5e48,2016-03-25 22:59:23 +0000,http://www.flipkart.com/fabhomedecor-fabric-do...,FabHomeDecor Fabric Double Sofa Bed,"[""Furniture >> Living Room Furniture >> Sofa B...",SBEEH3QGU7MFYJFY,32157.0,22646.0,"[""http://img6a.flixcart.com/image/sofa-bed/j/f...",False,FabHomeDecor Fabric Double Sofa Bed (Finish Co...,No rating available,No rating available,FabHomeDecor,"{""product_specification""=>[{""key""=>""Installati..."
2,f449ec65dcbc041b6ae5e6a32717d01b,2016-03-25 22:59:23 +0000,http://www.flipkart.com/aw-bellies/p/itmeh4grg...,AW Bellies,"[""Footwear >> Women's Footwear >> Ballerinas >...",SHOEH4GRSUBJGZXE,999.0,499.0,"[""http://img5a.flixcart.com/image/shoe/7/z/z/r...",False,Key Features of AW Bellies Sandals Wedges Heel...,No rating available,No rating available,AW,"{""product_specification""=>[{""key""=>""Ideal For""..."
3,0973b37acd0c664e3de26e97e5571454,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2F6HUZMQ6SJ,699.0,267.0,"[""http://img5a.flixcart.com/image/short/6/2/h/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
4,bc940ea42ee6bef5ac7cea3fb5cfbee7,2016-03-25 22:59:23 +0000,http://www.flipkart.com/sicons-all-purpose-arn...,Sicons All Purpose Arnica Dog Shampoo,"[""Pet Supplies >> Grooming >> Skin & Coat Care...",PSOEH3ZYDMSYARJ5,220.0,210.0,"[""http://img5a.flixcart.com/image/pet-shampoo/...",False,Specifications of Sicons All Purpose Arnica Do...,No rating available,No rating available,Sicons,"{""product_specification""=>[{""key""=>""Pet Type"",..."


In [3]:
# Shape of dataset
print(df.shape)

# Column names
print(df.columns)

# Data types
print(df.dtypes)

# Missing values
print(df.isnull().sum())

(20002, 15)
Index(['uniq_id', 'crawl_timestamp', 'product_url', 'product_name',
       'product_category_tree', 'pid', 'retail_price', 'discounted_price',
       'image', 'is_FK_Advantage_product', 'description', 'product_rating',
       'overall_rating', 'brand', 'product_specifications'],
      dtype='object')
uniq_id                     object
crawl_timestamp             object
product_url                 object
product_name                object
product_category_tree       object
pid                         object
retail_price               float64
discounted_price           float64
image                       object
is_FK_Advantage_product     object
description                 object
product_rating              object
overall_rating              object
brand                       object
product_specifications      object
dtype: object
uniq_id                       2
crawl_timestamp               2
product_url                   2
product_name                  2
product_category_tr

In [4]:
# Clean column names

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Check updated column names
print(df.columns)

Index(['uniq_id', 'crawl_timestamp', 'product_url', 'product_name',
       'product_category_tree', 'pid', 'retail_price', 'discounted_price',
       'image', 'is_fk_advantage_product', 'description', 'product_rating',
       'overall_rating', 'brand', 'product_specifications'],
      dtype='object')


In [5]:
# Check duplicate rows

print(df.duplicated().sum())

1


In [6]:
# Remove duplicate rows

df = df.drop_duplicates()

# Verify again
print(df.duplicated().sum())

# Check new shape
print(df.shape)

0
(20001, 15)


In [7]:
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percentage': missing_percent
})

print(missing_summary[missing_summary['missing_count'] > 0])

                         missing_count  missing_percentage
uniq_id                              1            0.005000
crawl_timestamp                      1            0.005000
product_url                          1            0.005000
product_name                         1            0.005000
product_category_tree                1            0.005000
pid                                  1            0.005000
retail_price                        79            0.394980
discounted_price                    79            0.394980
image                                4            0.019999
is_fk_advantage_product              1            0.005000
description                          3            0.014999
product_rating                       1            0.005000
overall_rating                       1            0.005000
brand                             5865           29.323534
product_specifications              15            0.074996


In [8]:
# Fill missing values

# Numeric columns -> median
df['retail_price'] = df['retail_price'].fillna(df['retail_price'].median())

df['discounted_price'] = df['discounted_price'].fillna(df['discounted_price'].median())

# Brand -> unknown
df['brand'] = df['brand'].fillna('unknown')

# Text/object columns -> unknown
text_columns = [
    'description',
    'image',
    'product_specifications',
    'product_name',
    'product_url',
    'product_category_tree',
    'pid'
]

for col in text_columns:
    df[col] = df[col].fillna('unknown')

# Ratings -> No Rating
df['product_rating'] = df['product_rating'].fillna('No Rating')

df['overall_rating'] = df['overall_rating'].fillna('No Rating')

# Boolean-like column
df['is_fk_advantage_product'] = (
    df['is_fk_advantage_product']
    .fillna(False)
)

# Timestamp
df['crawl_timestamp'] = (
    df['crawl_timestamp']
    .fillna('unknown')
)

# Check missing values again
print(df.isnull().sum())

uniq_id                    1
crawl_timestamp            0
product_url                0
product_name               0
product_category_tree      0
pid                        0
retail_price               0
discounted_price           0
image                      0
is_fk_advantage_product    0
description                0
product_rating             0
overall_rating             0
brand                      0
product_specifications     0
dtype: int64


C:\Users\jyoth\AppData\Local\Temp\ipykernel_1916\577850979.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [9]:
# Fill missing uniq_id

df['uniq_id'] = df['uniq_id'].fillna('unknown_id')

# Verify
print(df.isnull().sum())

uniq_id                    0
crawl_timestamp            0
product_url                0
product_name               0
product_category_tree      0
pid                        0
retail_price               0
discounted_price           0
image                      0
is_fk_advantage_product    0
description                0
product_rating             0
overall_rating             0
brand                      0
product_specifications     0
dtype: int64


In [10]:
# Convert timestamp column

df['crawl_timestamp'] = pd.to_datetime(
    df['crawl_timestamp'],
    errors='coerce'
)

# Convert ratings to numeric
df['product_rating'] = pd.to_numeric(
    df['product_rating'],
    errors='coerce'
)

df['overall_rating'] = pd.to_numeric(
    df['overall_rating'],
    errors='coerce'
)

# Check updated data types
print(df.dtypes)

uniq_id                                 object
crawl_timestamp            datetime64[ns, UTC]
product_url                             object
product_name                            object
product_category_tree                   object
pid                                     object
retail_price                           float64
discounted_price                       float64
image                                   object
is_fk_advantage_product                   bool
description                             object
product_rating                         float64
overall_rating                         float64
brand                                   object
product_specifications                  object
dtype: object


In [11]:
# Standardize text columns

text_columns = df.select_dtypes(include='object').columns

for col in text_columns:
    df[col] = df[col].str.strip().str.lower()

# Check sample
df.head()

,uniq_id,crawl_timestamp,product_url,product_name,product_category_tree,pid,retail_price,discounted_price,image,is_fk_advantage_product,description,product_rating,overall_rating,brand,product_specifications
0,c2d766ca982eca8304150849735ffef9,2016-03-25 22:59:23+00:00,http://www.flipkart.com/alisha-solid-women-s-c...,alisha solid women's cycling shorts,"[""clothing >> women's clothing >> lingerie, sl...",srteh2ff9kedefgf,999.0,379.0,"[""http://img5a.flixcart.com/image/short/u/4/a/...",False,key features of alisha solid women's cycling s...,NaN,NaN,alisha,"{""product_specification""=>[{""key""=>""number of ..."
1,7f7036a6d550aaa89d34c77bd39a5e48,2016-03-25 22:59:23+00:00,http://www.flipkart.com/fabhomedecor-fabric-do...,fabhomedecor fabric double sofa bed,"[""furniture >> living room furniture >> sofa b...",sbeeh3qgu7mfyjfy,32157.0,22646.0,"[""http://img6a.flixcart.com/image/sofa-bed/j/f...",False,fabhomedecor fabric double sofa bed (finish co...,NaN,NaN,fabhomedecor,"{""product_specification""=>[{""key""=>""installati..."
2,f449ec65dcbc041b6ae5e6a32717d01b,2016-03-25 22:59:23+00:00,http://www.flipkart.com/aw-bellies/p/itmeh4grg...,aw bellies,"[""footwear >> women's footwear >> ballerinas >...",shoeh4grsubjgzxe,999.0,499.0,"[""http://img5a.flixcart.com/image/shoe/7/z/z/r...",False,key features of aw bellies sandals wedges heel...,NaN,NaN,aw,"{""product_specification""=>[{""key""=>""ideal for""..."
3,0973b37acd0c664e3de26e97e5571454,2016-03-25 22:59:23+00:00,http://www.flipkart.com/alisha-solid-women-s-c...,alisha solid women's cycling shorts,"[""clothing >> women's clothing >> lingerie, sl...",srteh2f6huzmq6sj,699.0,267.0,"[""http://img5a.flixcart.com/image/short/6/2/h/...",False,key features of alisha solid women's cycling s...,NaN,NaN,alisha,"{""product_specification""=>[{""key""=>""number of ..."
4,bc940ea42ee6bef5ac7cea3fb5cfbee7,2016-03-25 22:59:23+00:00,http://www.flipkart.com/sicons-all-purpose-arn...,sicons all purpose arnica dog shampoo,"[""pet supplies >> grooming >> skin & coat care...",psoeh3zydmsyarj5,220.0,210.0,"[""http://img5a.flixcart.com/image/pet-shampoo/...",False,specifications of sicons all purpose arnica do...,NaN,NaN,sicons,"{""product_specification""=>[{""key""=>""pet type"",..."


In [12]:
# Fill rating NaN values with 0

df['product_rating'] = df['product_rating'].fillna(0)

df['overall_rating'] = df['overall_rating'].fillna(0)

# Verify
print(df[['product_rating', 'overall_rating']].isnull().sum())

product_rating    0
overall_rating    0
dtype: int64


In [13]:
# Create discount percentage column

df['discount_percentage'] = (
    (df['retail_price'] - df['discounted_price'])
    / df['retail_price']
) * 100

# Round values
df['discount_percentage'] = (
    df['discount_percentage']
    .round(2)
)

# Check sample
print(
    df[['retail_price',
        'discounted_price',
        'discount_percentage']].head()
)

   retail_price  discounted_price  discount_percentage
0         999.0             379.0                62.06
1       32157.0           22646.0                29.58
2         999.0             499.0                50.05
3         699.0             267.0                61.80
4         220.0             210.0                 4.55


In [14]:
# Check outliers using IQR method

Q1 = df['retail_price'].quantile(0.25)
Q3 = df['retail_price'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

outliers = df[
    (df['retail_price'] < lower_bound) |
    (df['retail_price'] > upper_bound)
]

print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("Outliers Count:", outliers.shape[0])

Lower Bound: -1311.0
Upper Bound: 3985.0
Outliers Count: 2063


In [15]:
# Create outlier flag column

df['price_outlier'] = np.where(
    (df['retail_price'] > upper_bound) |
    (df['retail_price'] < lower_bound),
    'yes',
    'no'
)

# Check counts
print(df['price_outlier'].value_counts())

price_outlier
no     17938
yes     2063
Name: count, dtype: int64


In [16]:
# Final dataset check

print("Final Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes)

print("\nSample Data:")
print(df.head())

Final Shape: (20001, 17)

Missing Values:
uniq_id                    0
crawl_timestamp            1
product_url                0
product_name               0
product_category_tree      0
pid                        0
retail_price               0
discounted_price           0
image                      0
is_fk_advantage_product    0
description                0
product_rating             0
overall_rating             0
brand                      0
product_specifications     0
discount_percentage        0
price_outlier              0
dtype: int64

Data Types:
uniq_id                                 object
crawl_timestamp            datetime64[ns, UTC]
product_url                             object
product_name                            object
product_category_tree                   object
pid                                     object
retail_price                           float64
discounted_price                       float64
image                                   object
is_fk_advantage_

In [17]:
# Fill remaining timestamp null

df['crawl_timestamp'] = df['crawl_timestamp'].fillna(
    pd.Timestamp('2016-01-01', tz='UTC')
)

# Final check
print(df.isnull().sum())

# Export cleaned dataset
df.to_csv(
    "cleaned_flipkart_dataset.csv",
    index=False
)

print("Cleaned dataset exported successfully!")

uniq_id                    0
crawl_timestamp            0
product_url                0
product_name               0
product_category_tree      0
pid                        0
retail_price               0
discounted_price           0
image                      0
is_fk_advantage_product    0
description                0
product_rating             0
overall_rating             0
brand                      0
product_specifications     0
discount_percentage        0
price_outlier              0
dtype: int64
Cleaned dataset exported successfully!


In [24]:
# Convert timestamp column to string

df = df.rename(columns={'crawl_timestamp': 'crawl_datetime'})

In [25]:
table_name = "flipkart_products"

df.to_sql(
    "flipkart_products",
    engine,
    if_exists="replace",
    index=False
)

print(f"✔ Data successfully loaded into table '{table_name}'")

✔ Data successfully loaded into table 'flipkart_products'


In [18]:
from sqlalchemy import create_engine
import urllib

# Your SQL Server details
server = r"JYOTHSNA"   # Use raw string for backslash
database = "flipkart_data_analysis"            # Your database name
driver = "ODBC Driver 17 for SQL Server"   # Must be installed on your machine

# Build connection string for Windows Authentication
connection_string = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
)

# Create engine
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={connection_string}")

print("✔ Successfully connected to SQL Server (Windows Authentication)!")

✔ Successfully connected to SQL Server (Windows Authentication)!


In [23]:
table_name = "flipkart_products"   # SQL table name you want to create

df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"✔ Data successfully loaded into table '{table_name}' in database 'flipkart_data_analysis.")

✔ Data successfully loaded into table 'flipkart_products' in database 'flipkart_data_analysis.
